# Chapter 5: Theoretical pricing models in code

Price an option the "simple approach" way: list possible prices at expiration, give each a probability, take the expected value of the option, then discount it.
Notes: [[chapters/05-theoretical-pricing-models]]

In [ ]:
import matplotlib.pyplot as plt

def expected_value(dist, f):
    """dist is a list of (price, probability). f turns a price into a payoff."""
    return sum(p * f(s) for s, p in dist)

def call_payoff(X):
    return lambda S: max(0, S - X)

def put_payoff(X):
    return lambda S: max(0, X - S)

def check_probabilities(dist):
    total = sum(p for s, p in dist)
    assert abs(total - 1) < 1e-9, f"probabilities add to {total}, not 1"

def growth(r, t):
    """Simple-interest growth factor 1 + r*t (stock-type settlement)."""
    return 1 + r * t

def theoretical_value(dist, X, kind, r, t):
    """Expected value of the option at expiration, then discounted to today."""
    check_probabilities(dist)
    payoff = call_payoff(X) if kind == "call" else put_payoff(X)
    return expected_value(dist, payoff) / growth(r, t)

## Expected value: the die and the roulette bet

In [ ]:
die = [(face, 1/6) for face in range(1, 7)]
print("Die: expected payout per roll =", round(expected_value(die, lambda s: s), 4))

roulette_ev = 36 / 38                      # one winning slot out of 38 pays $36
print("Roulette bet: expected value =", round(roulette_ev, 4), "dollars")
print("Casino edge on a $1.00 bet   =", round(1.00 - roulette_ev, 4), "dollars")

r, t = 0.12, 2/12                          # 12% a year, paid in two months
print("Book's rounding: 0.95 / (1 + 0.12*2/12) =", round(0.95 / growth(r, t), 4))
print("Exact 36/38 discounted               =", round(roulette_ev / growth(r, t), 4))

## The book's five distributions (underlying is a stock at 100, 12% interest, 2 months, no dividend)
The forward price is $100 \times 1.02 = 102$.

In [ ]:
r, t, X = 0.12, 2/12, 100
F = 100 * growth(r, t)
print("Forward price =", F)

distributions = {
    "Fig 5-1/5-2: equal chances":      [(80, .20), (90, .20), (100, .20), (110, .20), (120, .20)],
    "Fig 5-3: bunched around 100":     [(80, .10), (90, .20), (100, .40), (110, .20), (120, .10)],
    "Fig 5-4: bunched around 102":     [(82, .10), (92, .20), (102, .40), (112, .20), (122, .10)],
    "Fig 5-5: uneven, mean still 102": [(83, .06), (90, .15), (99, .39), (110, .33), (123, .07)],
}

print()
print(f"{'distribution':<34}{'mean':>8}{'E[100 call]':>13}{'theo value':>12}")
for name, dist in distributions.items():
    check_probabilities(dist)
    mean = expected_value(dist, lambda s: s)
    ev_call = expected_value(dist, call_payoff(X))
    print(f"{name:<34}{mean:>8.2f}{ev_call:>13.2f}{ev_call / growth(r, t):>12.2f}")

The book gets 6.00 (5.88), 4.00 (3.92), 5.40 (5.29) and 4.91 (4.81). The first two use a mean of 100, the last two use the forward price of 102.

## Check (our own, not in the book): call minus put
If the mean of the distribution is the forward price $F$, then for the same exercise price $X$:
$$E[\text{call}] - E[\text{put}] = F - X$$

In [ ]:
for name, dist in list(distributions.items())[2:]:
    ec = expected_value(dist, call_payoff(X))
    ep = expected_value(dist, put_payoff(X))
    print(f"{name}: E[call]={ec:.2f}  E[put]={ep:.2f}  difference={ec - ep:.2f}  F-X={F - X:.2f}")

## Same mean, wider spread: the call is worth more
Three prices, centred on 100: $100-d$, $100$, $100+d$ with probabilities 25%, 50%, 25%. Interest is ignored here (treated as zero) so only the spread changes.

In [ ]:
spread_rows = []
for d in (0, 5, 10, 15, 20, 30):
    dist = [(100 - d, .25), (100, .50), (100 + d, .25)]
    mean = expected_value(dist, lambda s: s)
    ev_call = expected_value(dist, call_payoff(100))
    spread_rows.append((d, mean, ev_call))
    print(f"d = {d:>2}: mean = {mean:.0f}, E[100 call] = {ev_call:.2f}")

## Figures

In [ ]:
import os
os.makedirs("../images", exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)
for ax, (name, dist) in zip(axes.ravel(), distributions.items()):
    prices = [s for s, p in dist]
    probs = [100 * p for s, p in dist]
    ev_call = expected_value(dist, call_payoff(X))
    ax.bar(prices, probs, width=4, color="#9ecae1", edgecolor="#3182bd")
    for s, p in zip(prices, probs):
        ax.text(s, p + 1, f"{p:.0f}%", ha="center", fontsize=9)
    ax.axvline(100, color="grey", ls=":", lw=1)
    ax.set_title(f"{name}\nE[100 call] = {ev_call:.2f}  (value today {ev_call / growth(r, t):.2f})", fontsize=10)
    ax.set_xlabel("Underlying price at expiration")
    ax.set_xlim(70, 135)
    ax.set_ylim(0, 50)
    ax2 = ax.twinx()
    xs = list(range(70, 136))
    ax2.plot(xs, [call_payoff(X)(s) for s in xs], color="#e6550d", lw=1.8)
    ax2.set_ylim(0, 40)
    ax2.set_yticks([])
axes[0, 0].set_ylabel("Probability (%)")
axes[1, 0].set_ylabel("Probability (%)")
fig.suptitle("Probabilities (bars) and the 100 call's payoff at expiration (orange line)", fontsize=12)
fig.tight_layout()
fig.savefig("../images/ch05-probability-distributions.png", dpi=130)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.2))
ds = [d for d, m, c in spread_rows]
cs = [c for d, m, c in spread_rows]
ax.bar([str(d) for d in ds], cs, color="#fdae6b", edgecolor="#e6550d")
for i, c in enumerate(cs):
    ax.text(i, c + 0.2, f"{c:.2f}", ha="center")
ax.set_xlabel("Spread d (prices are 100 - d, 100, 100 + d with 25%, 50%, 25%)")
ax.set_ylabel("Expected value of the 100 call")
ax.set_title("Same mean (100), wider spread: the call is worth more")
fig.tight_layout()
fig.savefig("../images/ch05-spread-and-call-value.png", dpi=130)
plt.close(fig)
print("saved two figures to ../images")

## Inputs in practice

In [ ]:
# Time: feed in calendar days; the model works with years
for days in (30, 60, 180):
    print(f"{days:>3} days = {days / 365:.4f} years")

# Which underlying price to feed in? The price at which you expect to make the hedge trade.
bid, ask = 75.20, 75.40
print()
print("Quote", bid, "-", ask, " midpoint", round((bid + ask) / 2, 2))
position_hedge = {
    "buy call":  ("sell underlying", bid),
    "sell put":  ("sell underlying", bid),
    "sell call": ("buy underlying",  ask),
    "buy put":   ("buy underlying",  ask),
}
for pos, (hedge, px) in position_hedge.items():
    print(f"{pos:<10} -> hedge: {hedge:<16} use price near {px}")